In [ ]:
import duckdb
from pprint import pprint
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer


from sklearn.model_selection import train_test_split
from sklearn.metrics import (auc, mean_squared_error, mean_absolute_error, r2_score)
from sklearn.metrics import classification_report, confusion_matrix # Optional: for evaluation


In [269]:
con = duckdb.connect("/home/etienne/projects/inatML/data/inat.duckdb")
df = con.execute("SELECT * FROM features.training").df()
con.close()

In [283]:
# Function for comparing different approaches
def score_dataset(X_train, X_test, y_train, y_test):
    model = LogisticRegression(max_iter=500)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test) # Returns the probability of each class
    accuracy = model.score(X_test, y_test)
    print(f"Accuracy: {accuracy}")
    # Confusion Matrix
    cm = confusion_matrix(y_test, preds)
    print(f"Confusion Matrix:\n{cm}")

    # Classification Report
    report = classification_report(y_test, preds)
    print(f"Classification Report:\n{report}")

    mae = mean_absolute_error(y_test, preds)
    print(f"MAE :\n{mae}")


In [271]:
train_size = 0.7
val_size = 0.15
test_size = 0.15
scaler = StandardScaler()
imputer = SimpleImputer()
cols_with_missing = ['positional_accuracy_m',
                     'oauth_application_id']



In [272]:
X = df[['photo_count',
        'has_description',
        'has_license',
        'positional_accuracy_m',
        'captive',
        'oauth_application_id',
        'geoprivacy_set',
        'has_license'

        ]]
y = df.pop('label')
X.head()

,photo_count,has_description,has_license,positional_accuracy_m,captive,oauth_application_id,geoprivacy_set,has_license
0,1,False,True,<NA>,True,2,False,True
1,3,True,False,6,True,3,False,False
2,1,False,True,8,True,3,False,True
3,1,False,True,<NA>,True,<NA>,False,True
4,2,False,False,3,True,2,False,False


In [273]:
imputed_X = pd.DataFrame(imputer.fit_transform(X))
imputed_X.columns = X.columns


In [274]:
imputed_X.info()

<class 'pandas.DataFrame'>
RangeIndex: 37924 entries, 0 to 37923
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   photo_count            37924 non-null  float64
 1   has_description        37924 non-null  float64
 2   has_license            37924 non-null  float64
 3   positional_accuracy_m  37924 non-null  float64
 4   captive                37924 non-null  float64
 5   oauth_application_id   37924 non-null  float64
 6   geoprivacy_set         37924 non-null  float64
 7   has_license            37924 non-null  float64
dtypes: float64(8)
memory usage: 2.3 MB


In [275]:
imputed_X['oauth_application_id'] = imputed_X['oauth_application_id'].astype('object')


In [276]:
# Get list of categorical variables
s = (imputed_X.dtypes == 'object')
object_cols = list(s[s].index)

print("Categorical variables:")
print(object_cols)

Categorical variables:
['oauth_application_id']


In [277]:
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output= False)
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', encoder, object_cols)
    ],
    remainder='passthrough'
).set_output(transform='pandas')

In [278]:
# Apply one-hot encoder to each column with categorical data
encoded_X = preprocessor.fit_transform(imputed_X)

In [279]:
X_train, X_test, y_train, y_test = train_test_split(encoded_X, y, test_size=0.25, random_state=0)

In [280]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [281]:
X_train_scaled

array([[-0.5188001 ,  1.47495939, -0.86808746, ..., -0.1713185 ,
         0.52365426,  0.52365426],
       [-0.5188001 ,  1.47495939, -0.86808746, ..., -0.1713185 ,
        -1.90965697, -1.90965697],
       [-0.5188001 , -0.67798477,  1.15195767, ...,  5.83708118,
         0.52365426,  0.52365426],
       ...,
       [-0.5188001 ,  1.47495939, -0.86808746, ..., -0.1713185 ,
         0.52365426,  0.52365426],
       [-0.5188001 , -0.67798477,  1.15195767, ..., -0.1713185 ,
         0.52365426,  0.52365426],
       [ 1.9275247 , -0.67798477, -0.86808746, ..., -0.1713185 ,
         0.52365426,  0.52365426]], shape=(28443, 14))

In [284]:
score_dataset(X_train_scaled, X_test_scaled, y_train, y_test)

Accuracy: 0.7015082797173294


NameError: name 'confusion_matrix' is not defined